# Tiền xử lý và Đóng gói dữ liệu Sketch-to-Fashion trên Kaggle

Notebook này thực hiện các bước:
1. Quét toàn bộ thư mục dữ liệu ảnh gốc (Real và Sketch) trên Kaggle.
2. Ghép cặp ảnh Real và Sketch dựa trên `filename_stem` trùng nhau.
3. Chọn ngẫu nhiên loại Sketch (HED, Pencil, Canny) dựa trên tỉ lệ mong muốn (Mặc định: HED 50%, Pencil 30%, Canny 20%).
4. Resize các cặp ảnh đã chọn về kích thước 256x256 và lưu vào cấu trúc thư mục sạch sẽ (`train_real/` và `train_sketch/`).
5. Tạo file ánh xạ `metadata.json` chứa các cặp ảnh tương ứng (sử dụng đường dẫn tương đối).
6. Nén (ZIP) thư mục kết quả để người dùng có thể tải về và upload ngược lại lên Kaggle như một Dataset tĩnh.

In [ ]:
import os
import json
import time
import random
import math
import shutil
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import cv2
from tqdm.auto import tqdm
from dataclasses import dataclass

### 1) Khởi tạo cấu hình và tham số hệ thống

In [ ]:
TARGET_CATEGORIES = [
    "10_dress",
    "8_skirt",
    "43_ruffle",
    "1_top__t_shirt__sweatshirt",
    "0_shirt__blouse",
    "4_jacket",
    "9_coat",
    "2_sweater",
    "3_cardigan",
    "5_vest",
    "6_pants",
    "7_shorts",
]
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

@dataclass
class PipelineConfig:
    """Configuration object for paired sketch-real samples."""

    real_root: str
    sketch_roots: Dict[str, str]
    categories: Tuple[str, ...] = tuple(TARGET_CATEGORIES)
    sketch_ratios: Dict[str, float] = None
    seed: int = 42

    def __post_init__(self):
        if self.sketch_ratios is None:
            self.sketch_ratios = {"hed": 0.5, "pencil": 0.3, "canny": 0.2}

        ratio_sum = sum(self.sketch_ratios.values())
        if not math.isclose(ratio_sum, 1.0, rel_tol=1e-6):
            raise ValueError(f"Sketch ratios must sum to 1.0, got {ratio_sum}")

### 2) Hàm phụ trợ quét và ghép cặp hình ảnh

In [ ]:
def list_category_images(root: Path, category: str) -> Dict[str, Path]:
    """Trả về từ điển map từ stem file sang đường dẫn ảnh đầy đủ cho một nhóm phân loại."""
    category_dir = root / category
    if not category_dir.exists():
        return {}

    return {
        p.stem: p
        for p in category_dir.iterdir()
        if p.suffix.lower() in IMG_EXTENSIONS
    }

def build_gan_pairs(config: PipelineConfig) -> Tuple[List[dict], dict]:
    """Tạo ra các cặp ảnh sketch-real đồng bộ theo tỉ lệ sketch được cấu hình."""
    print("\n[1/3] Quét các phân mục, đối chiếu các file Real - Sketch...")
    start_time = time.time()

    real_root = Path(config.real_root)
    sketch_roots = {k: Path(v) for k, v in config.sketch_roots.items()}
    rows: List[dict] = []
    skipped_no_sketch = 0

    for category in tqdm(config.categories, desc="Scanning categories"):
        real_images = list_category_images(real_root, category)
        if not real_images:
            continue

        sketch_index = {
            method: list_category_images(root, category)
            for method, root in sketch_roots.items()
        }

        for stem, real_path in real_images.items():
            candidates = {
                method: sketch_index[method][stem]
                for method in sketch_index
                if stem in sketch_index[method]
            }
            if not candidates:
                skipped_no_sketch += 1
                continue

            rows.append(
                {
                    "category": category,
                    "filename_stem": stem,
                    "real_path": str(real_path),
                    "sketch_candidates": candidates,
                    "available_methods": sorted(candidates.keys()),
                }
            )

    print(f"Khớp thành công {len(rows)} cặp. Bỏ qua {skipped_no_sketch} ảnh Real không tìm thấy Sketch.")
    print("\n[2/3] Phân chia phương pháp Sketch dựa trên tỉ lệ cấu hình...")

    rng = random.Random(config.seed)
    total_found = len(rows)
    methods = list(config.sketch_ratios.keys())
    targets = {m: int(total_found * config.sketch_ratios[m]) for m in methods}
    remaining = dict(targets)

    indices = list(range(total_found))
    rng.shuffle(indices)

    for idx in tqdm(indices, desc="Assigning sketch method"):
        row = rows[idx]
        available = row["available_methods"]
        preferred = [m for m in available if remaining.get(m, 0) > 0]
        method = rng.choice(preferred) if preferred else rng.choice(available)

        row["sketch_method"] = method
        row["sketch_path"] = str(row["sketch_candidates"][method])
        if method in remaining:
            remaining[method] -= 1

    final_rows = [
        {
            "category": r["category"],
            "filename_stem": r["filename_stem"],
            "real_path": r["real_path"],
            "sketch_path": r["sketch_path"],
            "sketch_method": r["sketch_method"],
        }
        for r in rows
    ]

    duration = time.time() - start_time
    print(f"Hoàn thành chuẩn bị cặp dữ liệu trong {duration:.2f} giây.")

    summary = {
        "num_pairs": len(final_rows),
        "sketch_method_counts": dict(Counter(r["sketch_method"] for r in final_rows)),
        "skipped_no_sketch": skipped_no_sketch,
    }
    return final_rows, summary

### 3) Định nghĩa hàm resize song song chỉ với các ảnh đã chọn

In [ ]:
def resize_single_image(args):
    """Hàm chạy song song: đọc, kiểm tra kích thước, resize và lưu ảnh."""
    src_path, dst_path, size = args
    try:
        img = cv2.imread(str(src_path))
        if img is not None:
            dst_path.parent.mkdir(parents=True, exist_ok=True)
            if img.shape[0] != size or img.shape[1] != size:
                img = cv2.resize(
                    img,
                    (size, size),
                    interpolation=cv2.INTER_LINEAR
                )
            cv2.imwrite(str(dst_path), img)
            return True
    except Exception as e:
        print(f"Lỗi khi xử lý {src_path}: {e}")
    return False

def preprocess_selected_pairs(
    gan_rows: List[dict],
    output_dir="/kaggle/working/preprocessed_data",
    size=256,
) -> List[dict]:
    """Lập danh sách và resize các ảnh thuộc danh sách đã chọn."""
    output_dir = Path(output_dir)
    real_out = output_dir / "train_real"
    sketch_out = output_dir / "train_sketch"

    print(f"Chuẩn bị tác vụ xử lý cho {len(gan_rows)} cặp ảnh...")
    tasks = []
    preprocessed_rows = []

    for row in gan_rows:
        cat = row["category"]
        stem = row["filename_stem"]
        
        real_src = Path(row["real_path"])
        sketch_src = Path(row["sketch_path"])
        
        # Giữ nguyên định dạng gốc của file (jpg, png,...)
        real_dst = real_out / cat / real_src.name
        sketch_dst = sketch_out / cat / sketch_src.name

        tasks.append((real_src, real_dst, size))
        tasks.append((sketch_src, sketch_dst, size))

        # Đường dẫn tương đối dùng để lưu vào metadata.json
        preprocessed_rows.append({
            "category": cat,
            "filename_stem": stem,
            "real_path": f"train_real/{cat}/{real_src.name}",
            "sketch_path": f"train_sketch/{cat}/{sketch_src.name}",
            "sketch_method": row["sketch_method"]
        })

    if tasks:
        max_workers = os.cpu_count() or 4
        print(f"Bắt đầu resize song song sử dụng {max_workers} processes...")
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            success_count = sum(
                tqdm(
                    executor.map(resize_single_image, tasks),
                    total=len(tasks),
                    desc="Resizing images",
                    unit="img",
                )
            )
        print(f"Hoàn thành. Đã lưu thành công {success_count:,}/{len(tasks):,} ảnh.")
    else:
        print("Không có ảnh nào cần xử lý.")

    return preprocessed_rows

### 4) Thực thi tiền xử lý, lưu metadata và nén ZIP dữ liệu

In [ ]:
# Đường dẫn dữ liệu gốc trên Kaggle
RAW_REAL_ROOT = "/kaggle/input/datasets/vunhuduc/train-fashion-no-blur/(1)_images_filtered_no_bg"
RAW_SKETCH_ROOTS = {
    "hed": "/kaggle/input/datasets/vunhuduc/train-sketch/fashion_category_filtered/(2)_sketch_hed",
    "pencil": "/kaggle/input/datasets/vunhuduc/train-sketch/fashion_category_filtered/(2)_sketch_pencil",
    "canny": "/kaggle/input/datasets/vunhuduc/train-sketch/fashion_category_filtered/(2)_sketch_canny",
}

# Thư mục chứa kết quả
OUTPUT_DIR = "/kaggle/working/preprocessed_data"

# Khởi tạo Config ghép cặp
PIPE_CFG = PipelineConfig(
    real_root=RAW_REAL_ROOT,
    sketch_roots=RAW_SKETCH_ROOTS,
)

# Bước 1: Quét và ghép cặp theo tỉ lệ từ dữ liệu gốc
raw_rows, summary = build_gan_pairs(PIPE_CFG)

# In tổng quát thông số trước khi resize
print("\n" + "=" * 30)
print("PIPELINE SUMMARY:")
print(f"- Tổng số cặp được chọn: {summary['num_pairs']:,}")
for method, count in summary["sketch_method_counts"].items():
    print(f"  + {method}: {count:,} mẫu")
print("=" * 30 + "\n")

# Bước 2: Resize song song các ảnh trong các cặp được chọn
preprocessed_rows = preprocess_selected_pairs(raw_rows, output_dir=OUTPUT_DIR, size=256)

# Bước 3: Lưu file metadata.json vào trong thư mục kết quả
metadata_file = Path(OUTPUT_DIR) / "metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(preprocessed_rows, f, indent=2, ensure_ascii=False)
print(f"Đã ghi file metadata tại {metadata_file}")

# Bước 4: Nén ZIP thư mục kết quả để tải về
print("\nBắt đầu đóng gói thư mục dữ liệu thành file ZIP...")
shutil.make_archive("/kaggle/working/preprocessed_data", "zip", OUTPUT_DIR)
print("\n=======================================================")
print("HOÀN THÀNH!")
print("File ZIP đã được tạo thành công tại: /kaggle/working/preprocessed_data.zip")
print("Hãy tải file ZIP này về máy tính của bạn và tải lên Kaggle dưới dạng một Dataset mới.")
print("=======================================================")